# PDF renamer

Scans a folder of PDFs, pulls out title, date, author, keywords and a short summary, and suggests
a new file name. Nothing on disk changes until you press **Rename ticked files**.

Run all cells, review the table and the list at the bottom, then press the button.

Everything runs locally. OCR only kicks in when a PDF has no usable text or title.

In [1]:
from pathlib import Path

FOLDER = Path("./test-data")
#Path.home() / "Downloads"
RECURSIVE = False
NAME_TEMPLATE = "{date} {title}.pdf"
MAX_TITLE_LEN = 80
DAY_FIRST = False  # how to read ambiguous dates like 03/04/2024
TEXT_PAGES = 3  # pages read for date, keywords and summary
OCR_ENABLED = True
OCR_DPI = 200
OUTPUT_DIR = Path("scan_history")  # timestamped scan results and rename logs

In [2]:
import csv
import re
import time
from collections import Counter
from datetime import date, datetime
from functools import lru_cache

import ipywidgets as widgets
import numpy as np
import pandas as pd
import pymupdf
from IPython.display import display
from rapidocr import RapidOCR

pymupdf.TOOLS.mupdf_display_errors(False)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

## Reading text

Every page becomes a list of `(text, size, y)` lines, where `y` is the position down the page
from 0 to 1. PDF text and OCR output share this shape, so the title logic works on both.

In [3]:
@lru_cache(maxsize=1)
def ocr_engine():
    return RapidOCR()


def page_lines(page):
    lines = []
    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            spans = [s for s in line["spans"] if s["text"].strip()]
            if spans:
                text = " ".join("".join(s["text"] for s in spans).split())
                lines.append((text, max(s["size"] for s in spans), line["bbox"][1] / page.rect.height))
    return lines


def ocr_lines(page):
    pix = page.get_pixmap(dpi=OCR_DPI, colorspace=pymupdf.csRGB, alpha=False)
    image = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    result = ocr_engine()(image[:, :, ::-1])  # RapidOCR expects BGR
    if result.txts is None:
        return []
    return [
        (" ".join(text.split()), float(box[3][1] - box[0][1]), float(box[0][1]) / pix.height)
        for box, text in zip(result.boxes, result.txts)
    ]


def lines_to_text(lines):
    return re.sub(r"-\n(?=[a-z])", "", "\n".join(line[0] for line in lines))

## Title

Order of preference:
1. PDF metadata, if it looks real and its words appear on the first page
2. The largest text near the top of the first page
3. Metadata that failed the first-page check
4. OCR of the first page
5. The cleaned-up file name

In [4]:
JUNK_TITLE = re.compile(
    r"""^\s*(untitled|document\d*|microsoft\s+\w+|powerpoint\s+presentation|slide\s*\d*|page\s*\d*
    |scan\w*|image\d*|print|title|default|new\s+document|pdf|none|null|abstract|contents|table\s+of\s+contents)\s*$
    |^\s*microsoft\s+(word|powerpoint|excel)\s*-
    |\.(docx?|pptx?|xlsx?|pdf|indd|tex|dvi|qxd|eps|html?|rtf|tiff?|jpe?g|png)\s*$
    |[\\/]{1}\w+[\\/]|^\s*/|^\s*[a-z]:\\
    |^[0-9a-f-]{16,}$
    |cid:|�|arxiv:|^\s*doi\b|https?:|www\.|©|copyright""",
    re.IGNORECASE | re.VERBOSE,
)


def usable_title(text):
    if not text or JUNK_TITLE.search(text):
        return False
    letters = sum(c.isalpha() for c in text)
    return letters >= 4 and letters / len(text) >= 0.5


def clean_title(text):
    text = re.sub(r'[<>:"/\\|?*\x00-\x1f]', " ", text)
    text = " ".join(text.split()).strip(" .-_,;")
    if text.isupper():
        text = text.title()
    if len(text) > MAX_TITLE_LEN:
        text = text[:MAX_TITLE_LEN].rsplit(" ", 1)[0].rstrip(" .-_,;")
    return text


def title_from_lines(lines):
    """Largest text in the top 60% of the page. Returns (title, stands_out_from_body_text)."""
    top = [line for line in lines if line[2] < 0.6 and usable_title(line[0])]
    if not top:
        return None, False
    weights = Counter()
    for text, size, _ in lines:
        weights[round(size, 1)] += len(text)
    body_size = weights.most_common(1)[0][0]
    largest = max(size for _, size, _ in top)
    if largest <= body_size * 1.1:
        return top[0][0], False
    start = next(i for i, line in enumerate(top) if line[1] >= largest * 0.95)
    parts = [top[start]]
    for line in top[start + 1 :]:
        if line[1] < largest * 0.95 or line[2] - parts[-1][2] > 0.08:
            break
        parts.append(line)
    return " ".join(part[0] for part in parts), True


def metadata_on_page(title, page_text):
    words = re.findall(r"\w{3,}", title.lower())
    page_words = set(re.findall(r"\w{3,}", page_text.lower()))
    return bool(words) and sum(w in page_words for w in words) / len(words) >= 0.5


def title_from_filename(path):
    stem = re.sub(r"[_+]+|%20", " ", path.stem)
    stem = re.sub(r"\s*\(\d+\)$", "", stem)
    return remove_dates(stem) or path.stem

## Date

Dates on the first pages win, with labelled ones ("Published:", "Date:") first. Dates of birth,
due dates and expiry dates are ignored. Next come dates in the file name, then the PDF creation
date, and finally the file's modified time, which in Downloads is usually just the download date.

In [5]:
MONTHS = "jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|june?|july?|aug(?:ust)?|sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?"
MONTH_NUMBERS = {m: i + 1 for i, m in enumerate("jan feb mar apr may jun jul aug sep oct nov dec".split())}
YEAR = r"(?P<y>(?:19|20)\d\d)"
ORD = r"(?:st|nd|rd|th)?"
DATE_PATTERNS = [
    rf"(?<!\d){YEAR}[-_./](?P<m>0?[1-9]|1[0-2])[-_./](?P<d>0?[1-9]|[12]\d|3[01])(?!\d)",
    rf"(?<!\d){YEAR}(?P<m>0[1-9]|1[0-2])(?P<d>0[1-9]|[12]\d|3[01])(?!\d)",
    rf"(?<!\d)(?P<d>\d{{1,2}}){ORD}[\s-]+(?:of\s+)?(?P<mon>{MONTHS})\.?,?[\s-]+{YEAR}(?!\d)",
    rf"(?<![a-z])(?P<mon>{MONTHS})\.?\s+(?P<d>\d{{1,2}}){ORD},?\s+{YEAR}(?!\d)",
    rf"(?<![\d/.-])(?P<a>\d{{1,2}})[/.-](?P<b>\d{{1,2}})[/.-]{YEAR}(?!\d)",
    rf"(?<![a-z\d])(?P<mon>{MONTHS})\.?,?\s+{YEAR}(?!\d)",
]
DATE_RE = [re.compile(p, re.IGNORECASE) for p in DATE_PATTERNS]
DATE_IGNORE = re.compile(r"birth|dob\b|d\.o\.b|expir|exp\.|due|valid|until|renew|deadline|born", re.IGNORECASE)
DATE_LABEL = re.compile(r"date|dated|published|issued|effective|statement|submitted|revised|as of", re.IGNORECASE)


def match_to_date(match):
    """Returns (date, 'YYYY-MM-DD' or 'YYYY-MM'), or None if the match isn't a real date."""
    parts = match.groupdict()
    year = int(parts["y"])
    if parts.get("a"):
        a, b = int(parts["a"]), int(parts["b"])
        if a > 12 or (DAY_FIRST and b <= 12):
            day, month = a, b
        else:
            month, day = a, b
    else:
        month = MONTH_NUMBERS[parts["mon"][:3].lower()] if parts.get("mon") else int(parts["m"])
        day = int(parts["d"]) if parts.get("d") else None
    try:
        found = date(year, month, day or 1)
    except ValueError:
        return None
    if not date(1980, 1, 1) <= found <= date.today():
        return None
    return found, found.strftime("%Y-%m-%d" if day else "%Y-%m")


def find_dates(text):
    """All dates in text as (position, label, is_labelled), skipping ignored contexts."""
    spans, found = [], []
    for pattern in DATE_RE:  # most specific first, so "Mar 2024" can't match inside "Mar 5, 2024"
        for match in pattern.finditer(text):
            if any(match.start() < end and start < match.end() for start, end in spans):
                continue
            spans.append(match.span())
            parsed = match_to_date(match)
            context = text[max(0, match.start() - 25) : match.start()]
            if parsed and not DATE_IGNORE.search(context):
                found.append((match.start(), parsed[1], bool(DATE_LABEL.search(context))))
    return sorted(found)


def remove_dates(text):
    for pattern in DATE_RE:
        text = pattern.sub(" ", text)
    return " ".join(text.split()).strip(" .-_,;")


def date_from_text(text):
    found = find_dates(text)
    labelled = [f for f in found if f[2]]
    full = [f for f in found if len(f[1]) == 10]
    best = (labelled or full or found or [None])[0]
    return best and best[1]


def date_from_metadata(meta):
    for key in ("creationDate", "modDate"):
        match = re.match(r"D:((?:19|20)\d\d)(\d\d)(\d\d)", meta.get(key) or "")
        if match:
            return "-".join(match.groups())
    return None


def pick_date(text, path, meta):
    if found := date_from_text(text):
        return found, "text"
    if found := date_from_text(path.stem):
        return found, "filename"
    if found := date_from_metadata(meta):
        return found, "metadata"
    return datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d"), "file modified"

## Summary and keywords

A simple extractive summary: the sentences with the most frequent words, in document order.
The stop words are English, so other languages get rougher results.

In [6]:
STOPWORDS = set(
    """a about above after again all also am an and any are as at be because been before being below between both
    but by can could did do does doing down during each few for from further had has have having he her here hers
    him his how i if in into is it its itself just may me might more most must my no nor not now of off on once
    only or other our ours out over own same she should so some such than that the their theirs them then there
    these they this those through to too under until up us very was we were what when where which while who whom
    why will with would you your yours et al fig figure page pages using used use""".split()
)


def summarize(text, sentences=3, keywords=6):
    flat = " ".join(text.split())
    words = [w for w in re.findall(r"[^\W\d_]{3,}", flat.lower()) if w not in STOPWORDS]
    if not words:
        return "", ""
    freq = Counter(words)
    candidates = [s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", flat) if 40 <= len(s) <= 400]

    def score(sentence):
        tokens = re.findall(r"[^\W\d_]{3,}", sentence.lower())
        return sum(freq[t] for t in tokens) / (len(tokens) + 5)

    top = set(sorted(candidates, key=score, reverse=True)[:sentences])
    summary = " ".join(s for s in candidates if s in top)
    return summary, ", ".join(w for w, _ in freq.most_common(keywords))

## Scanning

In [7]:
def has_garbled_text(text):
    return "�" in text or "(cid:" in text


def scan_pdf(path):
    row = {"path": path, "file": path.name, "error": None}
    try:
        with pymupdf.open(path) as doc:
            if doc.needs_pass:
                return row | {"error": "password protected"}
            meta = doc.metadata or {}
            pages = [doc[i] for i in range(min(TEXT_PAGES, doc.page_count))]
            all_lines = [page_lines(page) for page in pages]
            first_lines = all_lines[0] if all_lines else []
            first_text = lines_to_text(first_lines)
            used_ocr = False

            # No text layer, or an unreadable one: the whole page has to go through OCR
            if OCR_ENABLED and pages and (len(first_text.strip()) < 30 or has_garbled_text(first_text)):
                all_lines = [ocr_lines(page) for page in pages[:2]]
                first_lines, first_text = all_lines[0], lines_to_text(all_lines[0])
                used_ocr = True

            meta_title = (meta.get("title") or "").strip()
            meta_ok = usable_title(meta_title)
            page_title, stands_out = title_from_lines(first_lines)

            if meta_ok and metadata_on_page(meta_title, first_text):
                title, source = meta_title, "metadata"
            elif page_title and stands_out:
                title, source = page_title, "ocr" if used_ocr else "page"
            elif meta_ok:
                title, source = meta_title, "metadata (not on page)"
            elif page_title:
                title, source = page_title, "ocr (first line)" if used_ocr else "first line"
            else:
                title, source = None, None

            # Text was there but gave no title: maybe the title is an image, so try OCR on page 1
            if OCR_ENABLED and pages and not title and not used_ocr:
                ocr_title, _ = title_from_lines(ocr_lines(pages[0]))
                if ocr_title:
                    title, source, used_ocr = ocr_title, "ocr", True

            if not title:
                title, source = title_from_filename(path), "filename"

            text = "\n".join(lines_to_text(lines) for lines in all_lines)
            date_value, date_source = pick_date(text, path, meta)
            summary, keywords = summarize(text)
            title = clean_title(title) or clean_title(path.stem)
            sure = source in ("metadata", "page") and date_source != "file modified"
            return row | {
                "suggested": NAME_TEMPLATE.format(date=date_value, title=title),
                "confidence": "high" if sure else "low",
                "title": title,
                "title_source": source,
                "date": date_value,
                "date_source": date_source,
                "author": (meta.get("author") or "").strip(),
                "pages": doc.page_count,
                "ocr": used_ocr,
                "keywords": keywords,
                "summary": summary,
            }
    except Exception as exc:
        return row | {"error": f"{type(exc).__name__}: {exc}"}


def unique_names(rows):
    """Makes suggestions distinct from each other and from other files already in the folder."""
    taken = set()
    for row in rows:
        if row["error"]:
            continue
        folder, name = row["path"].parent, row["suggested"]
        stem, n = name[:-4], 2
        while (folder, name.lower()) in taken or (name.lower() != row["file"].lower() and (folder / name).exists()):
            name, n = f"{stem} ({n}).pdf", n + 1
        taken.add((folder, name.lower()))
        row["suggested"] = name
    return rows


pattern = "**/*.pdf" if RECURSIVE else "*.pdf"
paths = sorted(p for p in FOLDER.glob(pattern, case_sensitive=False) if p.is_file())
started = time.perf_counter()
rows = unique_names([scan_pdf(p) for p in paths])
scanned = pd.DataFrame(rows)

# Each scan gets its own timestamp, shared by its saved results and its rename log
RUN_STAMP = datetime.now().strftime("%Y-%m-%d_%H%M%S")
SCAN_CSV = OUTPUT_DIR / f"scan_{RUN_STAMP}.csv"
RENAME_LOG = OUTPUT_DIR / f"rename_log_{RUN_STAMP}.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
scanned.to_csv(SCAN_CSV, index=False, encoding="utf-8-sig")

print(f"Scanned {len(rows)} PDFs in {FOLDER} in {time.perf_counter() - started:.1f}s")
print(f"Saved results to {SCAN_CSV}")
if rows:
    print(f"OCR used on {int(scanned.get('ocr', pd.Series(dtype=bool)).fillna(False).sum())}, errors on {int(scanned['error'].notna().sum())}")

Scanned 24 PDFs in test-data in 1.4s
Saved results to scan_history\scan_2026-11-02_163309.csv
OCR used on 0, errors on 0


## Everything found

In [10]:
scanned

,path,file,error,suggested,confidence,title,title_source,date,date_source,author,pages,ocr,keywords,summary
0,"test-data\2011-12-01 Pit-bull reviewing, the pursuit of perfection and the victims of success.pdf","2011-12-01 Pit-bull reviewing, the pursuit of perfection and the victims of success.pdf",None,"2011-12-01 Pit-bull reviewing, the pursuit of perfection and the victims of success.pdf",high,"Pit-bull reviewing, the pursuit of perfection and the victims of success",page,2011-12-01,text,,3,False,"journals, referees, editors, authors, professional, papers","There are obvious reasons (rehearsed, for example, by Petsko [3]) to expect that professional scientists will make better decisions than professional editors on scientific papers. Robertson BMC Biology 2011, 9:84 http://www.biomedcentral.com/1741-7007/9/84 evaluations of scientific papers, than professional editors with a scientific background. Robertson BMC Biology 2011, 9:84 http://www.biomedcentral.com/1741-7007/9/84 Page 3 of 3"
1,test-data\2019-01-23 Second Harmonic Generation Imaging.pdf,2019-01-23 Second Harmonic Generation Imaging.pdf,None,2019-01-23 Second Harmonic Generation Imaging.pdf,high,Second Harmonic Generation Imaging,metadata,2019-01-23,filename,,161,False,"edited, imaging, series, cellular, second, harmonic","Second Harmonic Generation Imaging Edited by Francesco S. Campagnola FORTHCOMING The Fluorescent Protein Revolution edited by Richard N. Davidson Natural Biomarkers for Cellular Metabolism: Biology, Techniques, and Applications edited by Vladimir V."
2,test-data\2019-09-15 Opticalfiber-baseddispersionfor spectraldiscriminationin.pdf,2019-09-15 Opticalfiber-baseddispersionfor spectraldiscriminationin.pdf,None,2019-09-15 Opticalfiber-baseddispersionfor spectraldiscriminationin.pdf,high,Opticalfiber-baseddispersionfor spectraldiscriminationin,page,2019-09-15,text,,18,False,"optical, flim, time, madison, fluorescence, however","This is made possible using the time delay introduced in the fluorescence emission path by a dispersive optical fiber coupled to a detector operating in time-correlatedsingle-photoncountingmode.Thisadd-onspectralimplementationrequires onlyafewsimplemodificationstoanyexistingFLIMsystemandisconsiderablymorecostefficient compared to currently available spectral detectors. © The Authors. With a combined FLIM-HSI correlative microscopy scheme, each dimension (spectrum and lifetime)issimultaneouslyacquired.However,whenmultiplefluorophoresarepresentinthe sample,theresultinghistogramofemissioneventsisnolongerasimpleexponentialdecay or a single emission spectrum. Three channels were coupled by optical fibers to introduce temporal delays."
3,test-data\2022-10-06 Hardware-softwareco-designofanopen-source (2).pdf,2022-10-06 Hardware-softwareco-designofanopen-source (2).pdf,None,2022-10-06 Hardware-softwareco-designofanopen-source (2).pdf,high,Hardware-softwareco-designofanopen-source,page,2022-10-06,text,,19,False,"manager, image, analysis, acquisition, madison, multimodal","The acquisition workflow is built using Pycro-Manager, a Python package that bridges hardware control libraries of the Java-based open-source microscopysoftwareMicro-ManagerinaPythonenvironment. JournalofBiomedicalOptics 026501-2 February2023•Vol.28(2) Pycro-Manager31allows for complex acquisition and analysis pipelines that involve microscope control,dataacquisition,anddataanalysistobeprogrammedinasinglePythonenvironment. Evaluation of the DL-based image restoration methods on a pancreatic tissue microarray (TMA) slides set shows that the denoising and resolution enhancement model can improve the image signal-to-noise ratio (SNR) and image resolution, leading to shorter scan times while maintaining the necessary image quality for downstream image analysis."
4,test-data\2022-10-06 Hardware-softwareco-designofanopen-source.pdf,2022-10-06 Hardware-softwareco-designofanopen-source.pdf,None,2022-10-06 Hardware-softwareco-designofanopen-source.pdf,high,Hardware-softwareco-

In [8]:
display(scanned.drop(columns=["path"]))

,file,error,suggested,confidence,title,title_source,date,date_source,author,pages,ocr,keywords,summary
0,"2011-12-01 Pit-bull reviewing, the pursuit of perfection and the victims of success.pdf",None,"2011-12-01 Pit-bull reviewing, the pursuit of perfection and the victims of success.pdf",high,"Pit-bull reviewing, the pursuit of perfection and the victims of success",page,2011-12-01,text,,3,False,"journals, referees, editors, authors, professional, papers","There are obvious reasons (rehearsed, for example, by Petsko [3]) to expect that professional scientists will make better decisions than professional editors on scientific papers. Robertson BMC Biology 2011, 9:84 http://www.biomedcentral.com/1741-7007/9/84 evaluations of scientific papers, than professional editors with a scientific background. Robertson BMC Biology 2011, 9:84 http://www.biomedcentral.com/1741-7007/9/84 Page 3 of 3"
1,2019-01-23 Second Harmonic Generation Imaging.pdf,None,2019-01-23 Second Harmonic Generation Imaging.pdf,high,Second Harmonic Generation Imaging,metadata,2019-01-23,filename,,161,False,"edited, imaging, series, cellular, second, harmonic","Second Harmonic Generation Imaging Edited by Francesco S. Campagnola FORTHCOMING The Fluorescent Protein Revolution edited by Richard N. Davidson Natural Biomarkers for Cellular Metabolism: Biology, Techniques, and Applications edited by Vladimir V."
2,2019-09-15 Opticalfiber-baseddispersionfor spectraldiscriminationin.pdf,None,2019-09-15 Opticalfiber-baseddispersionfor spectraldiscriminationin.pdf,high,Opticalfiber-baseddispersionfor spectraldiscriminationin,page,2019-09-15,text,,18,False,"optical, flim, time, madison, fluorescence, however","This is made possible using the time delay introduced in the fluorescence emission path by a dispersive optical fiber coupled to a detector operating in time-correlatedsingle-photoncountingmode.Thisadd-onspectralimplementationrequires onlyafewsimplemodificationstoanyexistingFLIMsystemandisconsiderablymorecostefficient compared to currently available spectral detectors. © The Authors. With a combined FLIM-HSI correlative microscopy scheme, each dimension (spectrum and lifetime)issimultaneouslyacquired.However,whenmultiplefluorophoresarepresentinthe sample,theresultinghistogramofemissioneventsisnolongerasimpleexponentialdecay or a single emission spectrum. Three channels were coupled by optical fibers to introduce temporal delays."
3,2022-10-06 Hardware-softwareco-designofanopen-source (2).pdf,None,2022-10-06 Hardware-softwareco-designofanopen-source (2).pdf,high,Hardware-softwareco-designofanopen-source,page,2022-10-06,text,,19,False,"manager, image, analysis, acquisition, madison, multimodal","The acquisition workflow is built using Pycro-Manager, a Python package that bridges hardware control libraries of the Java-based open-source microscopysoftwareMicro-ManagerinaPythonenvironment. JournalofBiomedicalOptics 026501-2 February2023•Vol.28(2) Pycro-Manager31allows for complex acquisition and analysis pipelines that involve microscope control,dataacquisition,anddataanalysistobeprogrammedinasinglePythonenvironment. Evaluation of the DL-based image restoration methods on a pancreatic tissue microarray (TMA) slides set shows that the denoising and resolution enhancement model can improve the image signal-to-noise ratio (SNR) and image resolution, leading to shorter scan times while maintaining the necessary image quality for downstream image analysis."
4,2022-10-06 Hardware-softwareco-designofanopen-source.pdf,None,2022-10-06 Hardware-softwareco-designofanopen-source.pdf,high,Hardware-softwareco-designofanopen-source,page,2022-10-06,text,,19,False,"manager, image, analysis, acquisition, madison, multimodal","The acquisition workflow is built using Pycro-Manager, a Python package that bridges hardware control libraries of the Java-based open-source microscopysoftwareMicro-ManagerinaPythonenvironment. JournalofBiomedicalOptics 026501-2 February2023•Vol.28(2) Pycro-Manager31al

## Review and rename

High-confidence rows start ticked. Low confidence means the title came from OCR, the first line or
the file name, or the date is only the file's modified time. Edit any name in place, untick what
you want to keep, and press the button. Existing files are never overwritten, and every rename is
recorded in `scan_history/rename_log_<timestamp>.csv`, using the same timestamp as the scan.

In [9]:
def rename_files(pairs):
    """Renames (source, new name) pairs, never overwriting. Returns one message per pair."""
    messages = []
    new_log = not RENAME_LOG.exists()
    with RENAME_LOG.open("a", newline="", encoding="utf-8") as handle:
        log = csv.writer(handle)
        for source, name in pairs:
            name = name.strip()
            if not name.lower().endswith(".pdf"):
                name += ".pdf"
            if re.search(r'[<>:"/\\|?*\x00-\x1f]', name):
                messages.append(f"SKIPPED {source.name}: invalid characters in '{name}'")
                continue
            target = source.with_name(name)
            if target.exists():
                messages.append(f"SKIPPED {source.name}: '{name}' already exists")
                continue
            try:
                source.rename(target)
            except OSError as exc:
                messages.append(f"FAILED {source.name}: {exc}")
                continue
            if new_log:
                log.writerow(["time", "old_path", "new_path"])
                new_log = False
            log.writerow([datetime.now().isoformat(timespec="seconds"), source, target])
            handle.flush()
            messages.append(f"{source.name}  ->  {name}")
    if RENAME_LOG.stat().st_size == 0:  # nothing was renamed
        RENAME_LOG.unlink()
    return messages


review = []
review_rows = []
for row in rows:
    if row["error"] or row["suggested"] == row["file"]:
        continue
    tick = widgets.Checkbox(value=row["confidence"] == "high", indent=False, layout=widgets.Layout(width="24px"))
    name = widgets.Text(value=row["suggested"], layout=widgets.Layout(width="55%"))
    label = widgets.Label(f"[{row['confidence']}] {row['file']}", layout=widgets.Layout(width="40%"))
    review.append((row, tick, name))
    review_rows.append(widgets.HBox([tick, name, label]))

button = widgets.Button(description="Rename ticked files", button_style="warning")
output = widgets.Output()


def on_rename(_):
    chosen = [(row, name) for row, tick, name in review if tick.value and row["path"].exists()]
    messages = rename_files([(row["path"], name.value) for row, name in chosen])
    with output:
        print("\n".join(messages) or "Nothing ticked.")
    for row, name in chosen:
        if row["path"].with_name(name.value).exists():
            row["path"] = row["path"].with_name(name.value)
    button.disabled = True
    button.description = "Done - rerun the notebook to scan again"


button.on_click(on_rename)
display(widgets.VBox(review_rows + [button, output]))

## Undo

Reverses the renames in the newest log in `scan_history`, or in the log you pass in, newest rename
first. The log is kept with an `undone_` prefix. Uncomment to run.

In [ ]:
def undo_renames(log=None):
    log = log or max(OUTPUT_DIR.glob("rename_log_*.csv"), default=None)
    if log is None:
        print("No rename log found.")
        return
    with log.open(newline="", encoding="utf-8") as handle:
        entries = list(csv.DictReader(handle))
    for entry in reversed(entries):
        old, new = Path(entry["old_path"]), Path(entry["new_path"])
        if new.exists() and not old.exists():
            new.rename(old)
            print(f"{new.name}  ->  {old.name}")
        else:
            print(f"SKIPPED {new.name}")
    log.rename(log.with_name(f"undone_{log.name}"))


# undo_renames()